In [6]:
import os
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path

from utils.helper import _subpath_after
from utils.task_gen_helper import sample_df
from utils.task_gen_helper import move_images_to_local_folder
A_MAP = {
    0: 'b',
    1: 'x'
}

In [7]:
def prod_csvs(p, segment: str = "HvM_with_discfade", images_dir: Path = Path("images")) -> Path:
    """
    Convert the stored path `p` to a local path under `images/` when the
    segment is present in the original path.

    If `p` is NaN/None/empty, returns p unchanged (keeps original behavior).
    """
    if p is None:
        return p

    p_path = Path(p)
    sub_p = _subpath_after(p_path, segment=segment)
    if sub_p is not None:
        return images_dir / sub_p
    return p_path


def get_nback_csv(
        trial_df: pd.DataFrame,
        trials_per_session: int = 20,
        stim_per_trial: int = 6,
        out_fp: Optional[Path] = None,
        act_map: dict = A_MAP,
):
    """
    Build n-back style trials CSV rows from a `trial_df`.

    Returns:
        indices_per_trial: list of index-chains for each completed trial
        re_per_trial: list of act-chains for each completed trial
        nback_df: pandas.DataFrame with columns session, stim1..stimN, act2..actN
    """
    if stim_per_trial < 2:
        raise ValueError("stim_per_trial must be >= 2")

    rows = []
    indices_per_trial = []
    re_per_trial = []

    session = 0
    stim_chain = []
    idx_chain = []
    act_chain = []

    # iterate rows from input trial_df
    for i, row in trial_df.iterrows():
        if i % trials_per_session == 0:
            session += 1

        if len(stim_chain) == 0:
            stim_chain.extend([row["stim1_fp"], row["stim2_fp"]])
            idx_chain.extend([row["stim1"], row["stim2"]])
        else:
            stim_chain.append(row["stim2_fp"])
            idx_chain.append(row["stim2"])
        act_chain.append(row["same_category"])

        if len(stim_chain) == stim_per_trial:
            rows.append({
                "session": session,
                "stim1": stim_chain[0],
                "stim2": stim_chain[1],
                "stim3": stim_chain[2],
                "stim4": stim_chain[3],
                "stim5": stim_chain[4],
                "stim6": stim_chain[5],
                "act2": act_map[act_chain[0]],
                "act3": act_map[act_chain[1]],
                "act4": act_map[act_chain[2]],
                "act5": act_map[act_chain[3]],
                "act6": act_map[act_chain[4]],
            })
            indices_per_trial.append(list(idx_chain))
            re_per_trial.append(list(act_chain))

            # reset chains
            stim_chain = []
            idx_chain = []
            act_chain = []

    nback_df = pd.DataFrame(rows)

    if out_fp is not None:
        out_path = Path(out_fp)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        nback_df.to_csv(out_path, index=False)

    return indices_per_trial, re_per_trial, nback_df


def get_interdms_csv(
        trial_df: pd.DataFrame,
        trials_per_session: int = 20,
        out_fp: Optional[Path] = None,
        act_map: dict = A_MAP,
):
    """
    Build interleaved DMS trials by pairing rows from two independently shuffled copies.

    Returns:
        row_indices: list of [aa.stim1, bb.stim1, aa.stim2, bb.stim2] for each trial
        row_re: list of [aa.same_pos, bb.same_obj] for each trial
        df: DataFrame of rows
    """
    aa_df = trial_df.sample(frac=1).reset_index(drop=True)
    bb_df = trial_df.sample(frac=1).reset_index(drop=True)

    rows = []
    indices_per_trial = []
    re_per_trial = []
    session = 0

    for i, ((_, aa), (_, bb)) in enumerate(zip(aa_df.iterrows(), bb_df.iterrows())):
        if i % trials_per_session == 0:
            session += 1

        row = {
            "session": session,
            "stim1": aa['stim1_fp'],
            "stim2": bb['stim1_fp'],
            "stim3": aa['stim2_fp'],
            "stim4": bb['stim2_fp'],
            "act3": act_map[aa['same_pos']],
            "act4": act_map[bb['same_obj']],
        }
        rows.append(row)
        indices_per_trial.append([aa['stim1'], bb['stim1'], aa['stim2'], bb['stim2']])
        re_per_trial.append([aa['same_pos'], bb['same_obj']])

    interdms_df = pd.DataFrame(rows)

    if out_fp is not None:
        out_path = Path(out_fp)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        interdms_df.to_csv(out_path, index=False)

    return indices_per_trial, re_per_trial, interdms_df

In [8]:
good_seeds = [2025, 2026]

hvm_dir = '/Users/markbai/PycharmProjects/RNN_NatAbs/data/original'
N_STIMS = 100
N_OBJS = 16
GRID_SIZE = 3

img_loader, meta_data = sample_df(
    hvm_dir, N_OBJS, N_STIMS, GRID_SIZE,
    df_path='stim_subset.csv'
)
df = img_loader.df


removed 0 rows from the original behaviour_df
found 5760 local images after filtering
normalizing with stats: ([0, 0, 0], [1, 1, 1])
Loading subset csv at stim_subset.csv


In [9]:
for seed in good_seeds:
    trial_df = pd.read_csv(f'chain_set_{seed}.csv')
    print(trial_df[['same_category', 'same_obj', 'same_pos']].mean())

    trial_df['stim1_fp'] = trial_df['stim1_fp'].apply(
        lambda p: prod_csvs(p)
    )
    trial_df['stim2_fp'] = trial_df['stim2_fp'].apply(
        lambda p: prod_csvs(p)
    )

    trial_df.to_csv(f'chain_set_{seed}_local.csv', index=False)
    nback_indices, nback_re, nback_df = get_nback_csv(
        trial_df,
        trials_per_session=20,
        stim_per_trial=6,
        out_fp=f'nback_set_{seed}_local.csv',
    )
    interdms_indices, interdms_re, interdms_df = get_interdms_csv(
        trial_df,
        trials_per_session=20,
        out_fp=f'interdms_set_{seed}_local.csv',
    )
    nback_stim_re = [
    [
        int(
            int(df.loc[trial[i], 'cat_1b']) ==
            int(df.loc[trial[i + 1], 'cat_1b'])
        )
        for i in range(0, len(trial) - 1)
    ]
    for trial in nback_indices]
    stim_re = np.array(nback_stim_re)
    print((stim_re == np.array(nback_re)).all())

    interdms_stim_re = [
        [
            int(
                int(df.loc[trial[0], 'pos_1b']) ==
                int(df.loc[trial[2], 'pos_1b'])
            ),
            int(
                int(df.loc[trial[1], 'id_1b']) ==
                int(df.loc[trial[3], 'id_1b']) and
                int(df.loc[trial[1], 'cat_1b']) ==
                int(df.loc[trial[3], 'cat_1b'])
            )
        ]
        for trial in interdms_indices]
    interdms_stim_re = np.array(interdms_stim_re)
    print((interdms_stim_re == np.array(interdms_re)).all())

same_category    0.50
same_obj         0.38
same_pos         0.38
dtype: float64
True
True
same_category    0.49
same_obj         0.38
same_pos         0.42
dtype: float64
True
True


In [5]:
move_images_to_local_folder(
    df,
    Path("images"),
    '/Users/markbai/PycharmProjects/RNN_NatAbs/data/original/HvM_with_discfade',
)